# FR-UNet: случайные кадры из видео

[Модель и готовые веса авторов](https://github.com/lseventeen/FR-UNet),
[статья, 2022](https://doi.org/10.1109/JBHI.2022.3188710).
Веса обучены на сосудах сетчатки из DRIVE. Здесь смотрим их предсказания на наших видео.
Без эталонных масок это визуальная оценка, а не измерение точности.

Установить зависимости: `pip install -r requirements.txt`. Запускать ячейки сверху вниз.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import shutil
import sys
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from tqdm import tqdm

VIDEO = sorted(Path("sample-500/files").glob("*.avi"))[0]
N_FRAMES = 4
SEED = 42
SCALE = 0.25
THRESHOLD = 0.5

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
torch.set_num_threads(4)
print(VIDEO, DEVICE)

`VIDEO` — путь к любому видео, в том числе на флешке. `SCALE` — масштаб входа модели:
0.25 уменьшает стороны в 4 раза; можно попробовать 0.5 или 1.0, расход памяти вырастет.
Это наш параметр эксперимента, не требование авторов. `THRESHOLD` задаёт порог маски.

In [ ]:
COMMIT = "1856bc206055fdfccde184d0ebf9ac8c3207387d"
BASE = f"https://raw.githubusercontent.com/lseventeen/FR-UNet/{COMMIT}"
CACHE = Path(".cache/fr_unet")
CACHE.mkdir(parents=True, exist_ok=True)

for remote, name in [
    ("models/fr_unet.py", "fr_unet.py"),
    ("models/utils.py", "utils.py"),
    ("LICENSE", "LICENSE"),
    ("pretrained_weights/DRIVE/checkpoint-epoch40.pth", "drive.pth"),
]:
    path = CACHE / name
    if not path.exists():
        partial = path.with_suffix(path.suffix + ".part")
        with urlopen(f"{BASE}/{remote}") as response, partial.open("wb") as output:
            with tqdm.wrapattr(output, "write", total=int(response.headers.get("Content-Length", 0)) or None,
                               desc=name) as progress:
                shutil.copyfileobj(response, progress)
        partial.replace(path)

utils = CACHE / "utils.py"
utils.write_text(utils.read_text().replace("from timm.models.layers import trunc_normal_",
                                          "from torch.nn.init import trunc_normal_"))
(CACHE / "__init__.py").touch()
sys.path.insert(0, str(CACHE.parent.resolve()))
from fr_unet.fr_unet import FR_UNet

with torch.serialization.safe_globals([(dict, "bunch.Bunch")]):
    checkpoint = torch.load(CACHE / "drive.pth", map_location="cpu", weights_only=True)
model = FR_UNet().to(DEVICE).eval()
model.load_state_dict({key.removeprefix("module."): value for key, value in checkpoint["state_dict"].items()})
del checkpoint
print(f"FR-UNet: {sum(p.numel() for p in model.parameters()):,} параметров")

In [ ]:
cap = cv2.VideoCapture(str(VIDEO))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
indices = np.sort(np.random.default_rng(SEED).choice(frame_count, min(N_FRAMES, frame_count), replace=False))
frames = []
for index in indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))
    ok, frame = cap.read()
    if not ok:
        raise RuntimeError(f"Не удалось прочитать кадр {index}: {VIDEO}")
    frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
cap.release()
print(f"{frame_count} кадров, {fps:g} fps; выбраны {indices.tolist()}")

In [ ]:
@torch.inference_mode()
def predict(frame):
    height, width = frame.shape[:2]
    size = (max(8, round(width * SCALE)), max(8, round(height * SCALE)))
    image = Image.fromarray(frame).resize(size, Image.Resampling.LANCZOS).convert("L")
    gray = np.asarray(image, dtype=np.float32)
    gray = (gray - gray.min()) / max(float(np.ptp(gray)), 1.0)
    h, w = gray.shape
    padded = np.pad(gray, ((0, -h % 8), (0, -w % 8)), mode="edge")
    tensor = torch.from_numpy(padded)[None, None].to(DEVICE)
    probability = model(tensor).sigmoid()[0, 0, :h, :w].cpu().numpy()
    return cv2.resize(probability, (width, height), interpolation=cv2.INTER_LINEAR)

probabilities = []
seconds = []
for frame in tqdm(frames, desc="FR-UNet", unit="кадр"):
    start = time.perf_counter()
    probabilities.append(predict(frame))
    seconds.append(time.perf_counter() - start)
print("Секунд на кадр (включая подготовку, первый — с прогревом):", np.round(seconds, 3))

In [ ]:
fig, axes = plt.subplots(len(frames), 3, figsize=(15, 4 * len(frames)), squeeze=False)
for row, (index, frame, probability, elapsed) in enumerate(zip(indices, frames, probabilities, seconds)):
    mask = probability >= THRESHOLD
    overlay = frame.copy()
    overlay[mask] = (0.6 * overlay[mask] + 0.4 * np.array([0, 255, 100])).astype(np.uint8)
    axes[row, 0].imshow(frame)
    axes[row, 0].set_title(f"Кадр {index}, {index / fps:.2f} с")
    axes[row, 1].imshow(probability, cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set_title("Вероятность сосуда")
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title(f"Маска ≥ {THRESHOLD:g}; {elapsed:.2f} с")
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.show()

Перед коммитом очистить вывод ячеек: **Kernel → Restart Kernel and Clear Outputs of All Cells**.
Кадры, веса и БД остаются локально; этот ноутбук ничего не обучает и не отправляет данные в сеть.